# Descarga de ensamblados → Drive

Verifica los ensamblados candidatos contra NCBI y baja los confirmados a
`tesis/70_genomas/`, sin pasar por tu disco.

**Por qué acá:** la sesión de Claude tiene NCBI bloqueado por política, así que
no puede ni verificar ni bajar. Colab sí.

Toda la lógica vive en `scripts/fetch_genomes.sh` — este notebook solo lo
maneja. Corré antes `00_setup.ipynb`.

## Preámbulo: montar Drive y clonar el repo

El repo es público, así que el clon no necesita credenciales. **Los notebooks
llaman a los scripts del repo en vez de reimplementarlos**: el criterio de
selección de corridas y el de verificación de ensamblados tienen que vivir en
un solo lugar, o dejan de ser reproducibles.

In [59]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/tesis')
CLON  = pathlib.Path('/content/tesis')
assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'
print('Drive OK:', DRIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive OK: /content/drive/MyDrive/tesis


In [60]:
import shutil, subprocess

REPO = 'youkonskernel-afk/tesis'
URL_ANON = 'https://github.com/' + REPO + '.git'

_AYUDA = (
    "No pude clonar de forma anonima y no hay GITHUB_TOKEN en los Secrets.",
    "Dos salidas, cualquiera sirve:",
    "  a) hacer el repo publico: Settings -> General -> Change visibility",
    "  b) crear un PAT de solo lectura y guardarlo como GITHUB_TOKEN en el",
    "     panel de Secrets de Colab (la llave a la izquierda), habilitando",
    "     el acceso para este notebook.",
)


def _sin_token(txt, secreto):
    # git incluye la URL en sus mensajes de error, y esa URL lleva el token.
    return txt.replace(secreto, '***') if secreto else txt


def _actualizar():
    # El clon es un CACHE del repo, no un espacio de trabajo: nada de lo que se
    # escribe durante una corrida vive adentro (el ledger va a Drive). Por eso
    # reset --hard y no pull --ff-only: el pull falla apenas un archivo
    # versionado quede modificado, y fallaba sin hacer ruido, asi que la celda
    # seguia corriendo con el codigo viejo.
    for args in (['fetch', '--depth', '1', 'origin', 'HEAD'],
                 ['reset', '--hard', 'FETCH_HEAD']):
        r = subprocess.run(['git', '-C', str(CLON)] + args,
                           capture_output=True, text=True)
        if r.returncode != 0:
            return False
    return True


def _clonar_de_cero():
    # 1. Anonimo. Alcanza si el repo es publico.
    r = subprocess.run(['git', 'clone', '--depth', '1', URL_ANON, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return 'clon anonimo (el repo es publico)'

    # 2. Con token de los Secrets de Colab. Para repo privado.
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not tok:
        raise RuntimeError(chr(10).join(_AYUDA))

    url = 'https://x-access-token:' + tok + '@github.com/' + REPO + '.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('el clon con token fallo: ' + _sin_token(r.stderr, tok))

    # Sin esto el token queda escrito en .git/config dentro de la VM.
    subprocess.run(['git', '-C', str(CLON), 'remote', 'set-url', 'origin', URL_ANON],
                   capture_output=True, text=True)
    return 'clon con token (el repo es privado)'


def clonar():
    if CLON.exists():
        if _actualizar():
            return 'clon actualizado'
        # Un clon que no se puede actualizar es peor que no tenerlo: la celda
        # seguiria con codigo viejo sin avisar. Se tira y se clona de nuevo.
        shutil.rmtree(CLON)
    return _clonar_de_cero()


print(clonar())
print(subprocess.run(['git', '-C', str(CLON), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())


clon actualizado
880c025 8 de 9 ensamblados verificados; rhirr espera confirmar la cepa


In [61]:
import glob, os, subprocess, shutil

# CADA notebook de Colab corre en su propia VM: lo que instalo otro cuaderno no
# existe aca. Por eso esta celda esta en los cuatro y es idempotente: si las
# herramientas ya estan, no hace nada.
SRA_VER = '3.1.1'
URL = f'https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/{SRA_VER}/sratoolkit.{SRA_VER}-ubuntu64.tar.gz'


def sh(cmd, t=600):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)


def _en_path(ruta):
    if ruta and ruta not in os.environ['PATH']:
        os.environ['PATH'] = ruta + ':' + os.environ['PATH']


def instala_sra():
    # ya desempaquetado en esta VM de una corrida anterior de la celda
    c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
    if c:
        _en_path(c[0]);
    if shutil.which('prefetch') and shutil.which('vdb-validate'):
        return 'ya estaba'

    r = sh(f'wget -q -O /tmp/sra.tar.gz "{URL}"')
    if r.returncode == 0 and sh('tar -xzf /tmp/sra.tar.gz -C /opt').returncode == 0:
        c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
        if c:
            _en_path(c[0])
            return f'tarball oficial {SRA_VER}'

    # La version del tarball puede cambiar o desaparecer. apt es mas viejo, pero
    # aca solo se descarga y se valida: nada de esto entra en la tesis.
    if sh('apt-get -qq install -y sra-toolkit').returncode == 0 and shutil.which('prefetch'):
        return 'apt (version distinta del tarball)'

    raise RuntimeError(
        'No pude instalar sra-tools ni por tarball ni por apt. '
        'Revisa la version vigente en https://github.com/ncbi/sra-tools/wiki '
        'y ajusta SRA_VER.')


if not shutil.which('jq'):
    sh('apt-get -qq update'); sh('apt-get -qq install -y jq')
print('sra-tools:', instala_sra())

faltan = [b for b in ('prefetch', 'vdb-validate', 'jq', 'curl', 'git')
          if not shutil.which(b)]
if faltan:
    raise RuntimeError('faltan herramientas: ' + ', '.join(faltan))
print('herramientas OK:', 'prefetch vdb-validate jq curl git')

sra-tools: ya estaba
herramientas OK: prefetch vdb-validate jq curl git


In [62]:
GENOMAS = DRIVE / '70_genomas'
GENOMAS.mkdir(parents=True, exist_ok=True)

# Definido una sola vez para que cualquier celda de abajo lo use sin depender de
# haber corrido otra antes.
env_gen = dict(os.environ, GENOMES_DIR=str(GENOMAS))

print('destino:', GENOMAS)


destino: /content/drive/MyDrive/tesis/70_genomas


## 1. Qué falta

`candidato` = propuesto pero **no comprobado**. El script se niega a bajar esos
hasta que una persona los verifique. Un ensamblado equivocado no falla
ruidosamente: alinea peor y contamina la anotación.

In [63]:
!cd /content/tesis && ./scripts/fetch_genomes.sh estado

ORG      ESTADO      CONFIANZA  ASSEMBLY                         ACCESSION
rhirr    candidato   alta       ASM2621079v1                     GCF_026210795.1
sclsc    verificado  alta       ASM14694v2                       GCF_000146945.2
phypa    verificado  alta       Phypa V5                         GCF_000002425.5
cloro    verificado  alta       C_rosea_IK726                    GCA_902827195.2
prupe    verificado  alta       Prunus_persica_NCBIv2            GCF_000346465.2
maldo    verificado  alta       GDT2T_hap1                       GCF_042453785.1
gadmo    verificado  alta       gadMor3.0                        GCF_902167405.1
galga    verificado  alta       bGalGal1.mat.broiler.GRCg7b      GCF_016699485.2
maggi    verificado  alta       xbMagGiga1.1                     GCF_963853765.1

1 sin verificar. Corré: ./scripts/fetch_genomes.sh resolve


## 2. Verificar contra NCBI

Para cada organismo pregunta dos cosas: si el accession propuesto existe, y cuál
es el ensamblado de **referencia vigente** de la especie. La segunda importa más
que la primera — un accession puede existir y no ser el que corresponde.

**Leé la salida antes de seguir.**

In [64]:
!cd /content/tesis && ./scripts/fetch_genomes.sh resolve

== rhirr — Rhizophagus irregularis
   spec       : GCF_026210795.1  ASM2621079v1  confianza=alta
   candidato  : GCF_026210795.1  ASM2621079v1  nivel=Chromosome  N50=5.1 Mb  total=146.8 Mb  estado=current
   referencia : GCF_026210795.1  ASM2621079v1  nivel=Chromosome  N50=5.1 Mb  total=146.8 Mb
   >>> COINCIDE — el candidato ES la referencia vigente

== sclsc — Sclerotinia sclerotiorum
   spec       : GCF_000146945.2  ASM14694v2  confianza=alta
   candidato  : GCF_000146945.2  ASM14694v2  nivel=Scaffold  N50=1.6 Mb  total=38.3 Mb  cepa=1980  estado=current
   referencia : GCF_000146945.2  ASM14694v2  nivel=Scaffold  N50=1.6 Mb  total=38.3 Mb  cepa=1980
   >>> COINCIDE — el candidato ES la referencia vigente

== phypa — Physcomitrium patens
   spec       : GCF_000002425.5  Phypa V5  confianza=alta
   candidato  : GCF_000002425.5  Phypa V5  nivel=Chromosome  N50=17.4 Mb  total=472 Mb  estado=current
   referencia : GCF_000002425.5  Phypa V5  nivel=Chromosome  N50=17.4 Mb  total=472 Mb
 

## 2b. Ensamblados por cepa

Para cuando la especie **tiene** referencia pero los datos son de **otra cepa**,
y para sacar el accession de un ensamblado del que solo se conoce el nombre.

Quedó **una sola pregunta abierta de los 9**: el ensamblado de R1 para `rhirr`
(`ASM43914v3`) está **retirado por NCBI**, así que se pasa a la referencia
vigente `GCF_026210795.1` — Chromosome contra Scaffold, N50 5.1 Mb contra
0.3 Mb. Pero su record no trae campo de cepa, y el de R1 era **DAOM 197198**, el
aislado modelo. Los 146.8 Mb contra 136.7 son compatibles con un reensamblado
más completo del mismo material; hay que confirmarlo antes de verificar.


In [65]:
# (org, texto a buscar). '' en el texto = lista TODO, sin filtro.
CONSULTAS = [
    ('rhirr', '026210795'),  # la referencia vigente: que cepa declara?
    ('rhirr', 'DAOM'),       # y cuales son de DAOM 197198, el aislado de R1
]
TAXON = ''   # forzar otro nombre de especie (sinonimos); aplica a todas

for _org, _grep in CONSULTAS:
    cmd = ['./scripts/fetch_genomes.sh', 'cepas', _org]
    if TAXON:
        cmd += ['--taxon', TAXON]
    if _grep:
        cmd += ['--grep', _grep]
    print('$ ' + ' '.join(cmd))
    p = subprocess.Popen(cmd, cwd=CLON, env=env_gen, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    for ln in p.stdout:
        print(ln, end='')
    p.wait()
    print()


$ ./scripts/fetch_genomes.sh cepas rhirr --grep 026210795
== rhirr — Rhizophagus irregularis
     36 ensamblados; 2 coinciden con '026210795'
     GCA_026210795.1  ASM2621079v1  nivel=Chromosome  N50=5.1 Mb  total=146.8 Mb
     GCF_026210795.1  ASM2621079v1  nivel=Chromosome  N50=5.1 Mb  total=146.8 Mb

$ ./scripts/fetch_genomes.sh cepas rhirr --grep DAOM
== rhirr — Rhizophagus irregularis
     36 ensamblados; 14 coinciden con 'DAOM'
     GCA_020716725.1  ASM2071672v1  nivel=Chromosome  N50=5 Mb  total=147.2 Mb  cepa=DAOM-197198
     GCA_002897155.2  Rir_HGAP_ii_V2.1  nivel=Contig  N50=2.3 Mb  total=149.7 Mb  cepa=DAOM 181602=DAOM 197198
     GCA_000439145.3  ASM43914v3  nivel=Scaffold  N50=0.3 Mb  total=136.7 Mb  cepa=DAOM 197198
     GCF_000439145.1  ASM43914v3  nivel=Scaffold  N50=0.3 Mb  total=136.7 Mb  cepa=DAOM 197198
     GCA_003833115.1  ASM383311v1  nivel=Scaffold  N50=0.1 Mb  total=156.9 Mb  cepa=DAOM 240409
     GCA_003833045.1  ASM383304v1  nivel=Scaffold  N50=0 Mb  total=2

## 3. Confirmar

**Ya está hecho en `data/genomas.tsv`**: 8 de 9 quedaron `verificado` con los
resultados de `resolve`. Esta celda no hace falta esta vez — correrla con las
listas vacías no cambia nada, que es lo correcto.

Queda solo `rhirr` en `candidato`, esperando la respuesta de 2b sobre la cepa.


In [66]:
CONFIRMADOS = []   # p.ej. ['prupe', 'gadmo'] — solo los que diste por buenos
CORREGIR    = {}   # p.ej. {'galga': ('GCF_000002315.7', 'GRCg6a')}  accession, assembly

spec = CLON / 'data' / 'genomas.tsv'
lineas = spec.read_text().split('\n')
tocadas, vistos = [], set()

for i, ln in enumerate(lineas):
    if ln.startswith('#') or '\t' not in ln:
        continue
    f = ln.split('\t')
    org = f[0]
    if org not in CONFIRMADOS and org not in CORREGIR:
        continue
    vistos.add(org)
    if org in CORREGIR:
        acc, asm = CORREGIR[org]
        f[4], f[3] = acc, asm      # accession y assembly se corrigen juntos
    if org in CONFIRMADOS:
        f[5] = 'verificado'
    lineas[i] = '\t'.join(f)
    tocadas.append(f'  {org:8s} {f[4]:18s} {f[3]:32s} {f[5]}')

# Un typo en CONFIRMADOS no puede pasar en silencio: hoy no haria nada y la
# celda diria que todo salio bien.
faltan = sorted((set(CONFIRMADOS) | set(CORREGIR)) - vistos)
assert not faltan, f'estos no estan en genomas.tsv: {faltan}'

spec.write_text('\n'.join(lineas))
print('\n'.join(tocadas) if tocadas else '(nada que cambiar; no se va a bajar nada)')


(nada que cambiar; no se va a bajar nada)


## 4. Bajar a Drive

In [67]:
!cd /content/tesis && GENOMES_DIR=/content/drive/MyDrive/tesis/70_genomas ./scripts/fetch_genomes.sh fetch

SALTO rhirr: 'candidato' (confianza alta). Verificá: ./scripts/fetch_genomes.sh resolve rhirr
== sclsc: bajando GCF_000146945.2 (ASM14694v2)
   12M  sha256 -> data/genomas.sha256
== phypa: bajando GCF_000002425.5 (Phypa V5)
   145M  sha256 -> data/genomas.sha256
== cloro: bajando GCA_902827195.2 (C_rosea_IK726)
   22M  sha256 -> data/genomas.sha256
== prupe: ya está en /content/drive/MyDrive/tesis/70_genomas/prupe/GCF_000346465.2.fna.gz
== maldo: ya está en /content/drive/MyDrive/tesis/70_genomas/maldo/GCF_042453785.1.fna.gz
== gadmo: ya está en /content/drive/MyDrive/tesis/70_genomas/gadmo/GCF_902167405.1.fna.gz
== galga: ya está en /content/drive/MyDrive/tesis/70_genomas/galga/GCF_016699485.2.fna.gz
== maggi: ya está en /content/drive/MyDrive/tesis/70_genomas/maggi/GCF_963853765.1.fna.gz

bajados=3 saltados=1
Subir a Drive:  ./scripts/drive_push.sh genomas --go
Y commitear:    git add data/genomas.sha256


## 5. Verificar lo que está bajado

Chequea los **9** organismos contra los archivos, no contra la spec ni el
ledger: que el FASTA exista, que arranque con `>`, y que su `sha256` coincida
con `data/genomas.sha256`. Si coincide, el archivo es idéntico byte a byte al
que se bajó —y como `fetch` lo escribió con gzip, la integridad del stream va
implícita, así que no hace falta leerlo dos veces. Cuando **no** hay entrada en
el ledger, ahí sí corre `gzip -t`, que es lo que detecta un archivo cortado.

Sale con código distinto de cero si algo falla, así que sirve como control antes
de arrancar el alineamiento.

`--rapido` solo mira existencia y la primera línea, sin leer el archivo entero
—que sobre el FUSE de Drive son ~1 GB por genoma.


In [68]:
ORG    = ''     # '' = los 9, o 'prupe', 'cloro', ...
RAPIDO = False  # True = no verifica checksums

cmd = ['./scripts/fetch_genomes.sh', 'verificar']
if ORG:
    cmd.append(ORG)
if RAPIDO:
    cmd.append('--rapido')

print(' '.join(cmd))
p = subprocess.Popen(cmd, cwd=CLON, env=env_gen, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for ln in p.stdout:
    print(ln, end='')
print('exit =', p.wait())


./scripts/fetch_genomes.sh verificar
arbol de genomas: /content/drive/MyDrive/tesis/70_genomas

ORG     ESTADO      ACCESSION          TAMANO    CHEQUEO
rhirr   candidato   GCF_026210795.1    -         FALTA — corre: ./scripts/fetch_genomes.sh fetch rhirr
sclsc   verificado  GCF_000146945.2    12.5 MB   OK
phypa   verificado  GCF_000002425.5    151.2 MB  OK
cloro   verificado  GCA_902827195.2    22.2 MB   OK
prupe   verificado  GCF_000346465.2    72.1 MB   OK
maldo   verificado  GCF_042453785.1    209.1 MB  OK
gadmo   verificado  GCF_902167405.1    200.4 MB  OK
galga   verificado  GCF_016699485.2    337.9 MB  OK
maggi   verificado  GCF_963853765.1    179.9 MB  OK

ok=8  con problemas=0  sin bajar=1  sin respaldo=0
exit = 1


## 6. Cerrar el círculo con git

El clon es efímero: se pierde al cerrar la sesión. Copiá esta salida al repo y
commiteala — **el checksum versionado es lo que deja constancia de qué genoma se
usó**, porque el que queda al lado del FASTA en Drive no prueba nada: quien
reemplace el genoma reemplaza el checksum con él.

In [69]:
spec = CLON / 'data' / 'genomas.tsv'
led = CLON / 'data' / 'genomas.sha256'
print('--- data/genomas.sha256 ---')
print(led.read_text() if led.exists() else '(vacio: no se bajo nada)')
print('--- data/genomas.tsv ---')
print(spec.read_text())


--- data/genomas.sha256 ---
org	accession	assembly	sha256	fecha_utc
cloro	GCA_902827195.2	C_rosea_IK726	68d46131ea7363d3d26007e1b0411042e033c40897f03df37389f6318b6f535a	2026-09-17
gadmo	GCF_902167405.1	gadMor3.0	6ab4c34995f75e24f9f3d3aa96f44b76e25069fb26775e3a93f8e1016f083bc0	2026-09-17
galga	GCF_016699485.2	bGalGal1.mat.broiler.GRCg7b	a1715b623b905567d12beba6a150b637f1d82f2f60c7ddf13080144041a5bc7a	2026-09-17
maggi	GCF_963853765.1	xbMagGiga1.1	cf64e22128c820a14c81c6054361f3053e2a299563f6c96f5f4dc91e8f7f71d0	2026-09-17
maldo	GCF_042453785.1	GDT2T_hap1	6a79b0ef7b0c822c6443a7dc6d1104c651ebd31158cc804412c73b0920860dad	2026-09-17
phypa	GCF_000002425.5	Phypa V5	84dd9413e770682380321428cb6af2ccc52e87b02c503afd12b52b857c120b79	2026-09-17
prupe	GCF_000346465.2	Prunus_persica_NCBIv2	2b2290bbc405c1feea4f9be4993baedc62aef6c904cfeb2669a52ca3be42bbdb	2026-09-17
sclsc	GCF_000146945.2	ASM14694v2	540c965242beb66c3c16b28f6e385d0214c1715eab133a96d5d9ffbc5c2c71b3	2026-09-17

--- data/genomas.tsv ---
# En